# 🧠 Neuron ASD — Interactive Explorer (Colab)

Fully **Colab-native** (form controls — no ipywidgets). Predicted effects of receptor /
neuromodulator modulations that move an autistic subject's resting EEG toward a
typically-developing (TD) reference.

**Workflow:**
1. **Setup** — install the engine (+MNE). Does **not** install TensorFlow (Colab's build is used as-is).
2. **Load EEG** — ASD subjects (1, 3, or any number) + a **TD pool** (loaded once, kept across ASD re-uploads). Any format via MNE (`.set`+`.fdt`, `.edf`, `.bdf`, `.vhdr`, `.fif`, …). TD simulator available (least reliable).
3. **Engine + TD reference** — classic (paper) or realistic (tissue-filter extension). For real EEG, *Realistic* is the 1/f-coherent choice. Builds and caches the TD reference once.
3b. **Accelerator** *(optional, before evaluations)* — train the CNN surrogate only if you need to speed up **large** batches. Not needed for interactive use.
4. **Modulate manually** — set knobs, run, get the predicted effect **plus a clinical-style note**. Repeat freely with different settings. Modulations are translated exactly as in the paper, so results here match the Step-5 recommendations.
5. **Recommendation** *(optional)* — the model's own best per-subject modulations (reliable moves only, ≥60% confidence), with a clinical-style note. Ensemble cached after the first (~3–4 min) run.

> **Model-based predictions for hypothesis generation — not clinical prescriptions.**
> Repository & DOI: https://doi.org/10.5281/zenodo.21581231


In [ ]:
#@title 🧠 Neuron ASD — Interactive Explorer · Step 1: Setup { display-mode: "form" }
#@markdown Run once to install the validated Neuron ASD engine (~1 min).
#@markdown
#@markdown **What this tool does:** for each autistic (ASD) subject you provide, it shows the
#@markdown *predicted effect* of receptor / neuromodulator modulations that move the subject's
#@markdown resting-EEG spectral profile toward a typically-developing (TD) reference.
#@markdown
#@markdown > Model-based predictions for **hypothesis generation** — not clinical prescriptions.
#@markdown > Repository & DOI: https://doi.org/10.5281/zenodo.21581231

import subprocess, sys, os, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

def _pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

print("Installing Neuron ASD (validated engine) + dependencies …")
# NOTE: we deliberately do NOT install tensorflow/tensorflow-cpu — Colab ships a working build,
# and pip-installing over it corrupts the native libraries (undefined-symbol crashes).
_pip("git+https://github.com/arianadelg/neuron-asd-tool.git")
_pip("matplotlib", "numpy", "scipy", "fooof", "mne")   # mne reads .set/.edf/.bdf/.vhdr/.fif/…

import numpy as np
import matplotlib.pyplot as plt
import neuron_asd.engine as N
from neuron_asd import app

# ---- global state shared across cells (persists while the runtime is alive) ----
ASD_SIGNALS = {}     # name -> raw 1D signal (conditioned)
TD_SIGNALS  = {}     # name -> raw 1D signal (conditioned)  [the expensive pool — load once]
CACHE = {            # everything we compute once and reuse, so re-runs are instant
    "TD_BANDS": None, "TD_EXP": None, "td_signature": None,
    "subj_feats": {}, # name -> (bands, exp)   cached per subject
}
STATE = {"engine": "classic", "td_source": "uploaded_pool"}

BANDS = list(N.BANDS.keys()) if hasattr(N.BANDS, "keys") else list(N.BANDS)
TARGETS = list(N.TARGET_KEYS)
NEUROMODS = list(N.NEUROMOD_KEYS)
MOD_ORDER = list(N.MOD_ORDER)
FS = N.FS

# quick check that TensorFlow (Colab's build) is importable — only needed IF you later
# choose to use the optional surrogate accelerator. Not required for the main workflow.
try:
    import tensorflow as _tf
    _TF_OK = True
    print(f"  (optional accelerator available: TensorFlow {_tf.__version__})")
except Exception:
    _TF_OK = False
    print("  (TensorFlow not importable — the optional accelerator is disabled; the main"
          " workflow does not need it)")

print("\n✓ Neuron ASD ready.")
print("  Targets:", ", ".join(TARGETS))
print("  Neuromodulators:", ", ".join(NEUROMODS))
print("\nNext: Step 2 to load EEG (ASD subjects + TD pool).")

In [ ]:
#@title 📂 Step 2: Load EEG data (ASD subjects + TD pool) { display-mode: "form" }
#@markdown Upload **1, 3 or any number** of ASD subjects, and a **TD pool** (≈20 recommended).
#@markdown Any format the main tool reads is accepted via MNE: EEGLAB `.set`(+`.fdt`), `.edf`,
#@markdown `.bdf`, `.gdf`, BrainVision `.vhdr`(+`.eeg`/`.vmrk`), `.fif`, `.cnt`, `.mff`,
#@markdown plus `.npy`/`.csv`/`.txt`. For multi-file formats, **select all parts together**.
#@markdown
#@markdown 💡 **The TD pool is loaded once and kept.** After the first run you can re-upload only
#@markdown the ASD subjects (leave *reload_TD_pool* unticked) — the TD reference stays cached.

reload_ASD_subjects = True  #@param {type:"boolean"}
reload_TD_pool      = True  #@param {type:"boolean"}
TD_reference_source = "Upload / keep a real TD pool (recommended)"  #@param ["Upload / keep a real TD pool (recommended)", "Use the TD simulator (LEAST reliable)"]

import os, numpy as np
UPLOAD_DIR = "/content/neuron_asd_uploads"; os.makedirs(UPLOAD_DIR, exist_ok=True)
_PRIMARY_EXT = ('.set','.edf','.bdf','.gdf','.vhdr','.fif','.fif.gz','.cnt','.cdt','.mff','.raw','.nxe')
_PLAIN_EXT   = ('.npy','.csv','.txt','.tsv')

def _save_uploads(label):
    from google.colab import files
    print(f"\n⬆  Select {label} file(s) (include every part of multi-file formats)…")
    up = files.upload()
    paths = []
    for fn, content in up.items():
        p = os.path.join(UPLOAD_DIR, fn)
        open(p, "wb").write(content); paths.append(p)
    return paths

def _signal_from_path(path):
    low = path.lower()
    if low.endswith(_PLAIN_EXT):
        try:
            arr = np.load(path) if low.endswith('.npy') else \
                  np.loadtxt(path, delimiter=',' if low.endswith(('.csv','.tsv')) else None)
            arr = np.asarray(arr, float)
            if arr.ndim > 1:
                arr = arr.mean(axis=0) if arr.shape[0] < arr.shape[1] else arr.mean(axis=1)
            arr = arr.ravel()
            return arr if arr.size >= FS*2 else None
        except Exception as e:
            print(f"  ! {os.path.basename(path)}: {e}"); return None
    try:
        raw = app._load_raw(path)
        roi = getattr(app, "CENTRAL_ROI", None) or raw.ch_names
        sig, have = app._roi_signal(raw, roi)
        if sig is None:
            with app._suppress_output():
                if abs(raw.info['sfreq'] - FS) > 1e-6: raw.resample(FS, verbose='ERROR')
                raw.filter(0.5, 80.0, verbose='ERROR')
            sig = raw.get_data().mean(axis=0)
            n = int(getattr(app, "WINDOW_SECONDS", 120) * FS)
            sig = sig[:n] if len(sig) >= n else sig
        return np.asarray(sig, float) if sig is not None and len(sig) >= FS*2 else None
    except Exception as e:
        print(f"  ! {os.path.basename(path)}: {e}"); return None

def _load_group(label):
    paths = _save_uploads(label)
    primaries = [p for p in paths if p.lower().endswith(_PRIMARY_EXT + _PLAIN_EXT)]
    if not primaries:
        print("  (no primary EEG file detected — did you include the .set/.edf/.vhdr/… file?)")
    out = {}
    for p in primaries:
        sig = _signal_from_path(p)
        nm = os.path.splitext(os.path.basename(p))[0]
        if sig is not None:
            out[nm] = sig; print(f"  ✓ {os.path.basename(p)}  ({sig.size/FS:.1f}s)")
        else:
            print(f"  ✗ {os.path.basename(p)} skipped")
    return out

# --- ASD subjects: reloading them invalidates only their cached features ---
if reload_ASD_subjects:
    ASD_SIGNALS.clear(); CACHE["subj_feats"].clear()
    ASD_SIGNALS.update(_load_group("ASD subject"))
print(f"\nASD subjects loaded: {len(ASD_SIGNALS)}  {list(ASD_SIGNALS.keys())}")

# --- TD reference: the expensive pool. Load once; keep across ASD re-uploads. ---
STATE["td_source"] = "simulator" if TD_reference_source.startswith("Use the TD simulator") else "uploaded_pool"
if STATE["td_source"] == "simulator":
    print("\n⚠  Using the TD SIMULATOR — the LEAST reliable option: the model-based TD reference is")
    print("   invalid on the aperiodic (1/f) axis (see the paper). Prefer a real TD pool.")
    TD_SIGNALS.clear(); CACHE["TD_BANDS"] = None   # force rebuild in Step 3
else:
    if reload_TD_pool or len(TD_SIGNALS) == 0:
        TD_SIGNALS.clear(); CACHE["TD_BANDS"] = None
        TD_SIGNALS.update(_load_group("TD reference"))
    else:
        print("\n(keeping the previously loaded TD pool — not re-uploading)")
    print(f"TD pool: {len(TD_SIGNALS)} recordings")
    if 0 < len(TD_SIGNALS) < 20:
        print(f"⚠  Only {len(TD_SIGNALS)} TD recordings; the E/I read-out stabilizes around ~20. "
              "Treat results as provisional.")

print("\nNext: Step 3 to build the TD reference (once) and prepare subjects.")

In [ ]:
#@title ⚙️ Step 3: Engine + build TD reference (cached) { display-mode: "form" }
#@markdown Pick the engine and run **once** after loading data. This builds the TD reference and
#@markdown pre-computes each subject's features, so Steps 4 and 5 are then **instant and repeatable**.
#@markdown Re-run this only if you change the engine or reload data.

engine_mode = "Classic (paper-validated)"  #@param ["Classic (paper-validated)", "Realistic (tissue filter, extension)"]
#@markdown *Classic* reproduces the paper exactly. *Realistic* adds the 1/f tissue-filter extension.
#@markdown
#@markdown 💡 **Which engine for real EEG?** Real EEG carries its own 1/f (aperiodic) physics.
#@markdown In *Realistic*, the simulated TD reference and the modulation responses also carry that
#@markdown 1/f structure, so they live on the **same physical scale** as your real recordings —
#@markdown this is the more coherent choice when comparing against real EEG. In *Classic*, the
#@markdown simulated reference is aperiodically flat (the "invalid on the aperiodic axis" finding of
#@markdown the paper), so distances mix two scales. Use *Classic* to reproduce the paper's validated
#@markdown numbers; prefer *Realistic* when interpreting modulations against your own real EEG.

import numpy as np

STATE["engine"] = "realistic" if engine_mode.startswith("Realistic") else "classic"
_BETA = 1.7 if STATE["engine"] == "realistic" else None

def _feats(signal):
    """Validated feature pipeline (Section 2.4): 5-band profile (dB) + aperiodic exponent."""
    band_vec, exp, r2 = app._features_from_signal(signal)
    return np.asarray(band_vec, float), exp

def _td_signature():
    # identifies the current TD reference so we know when the cache is stale
    return (STATE["td_source"], STATE["engine"], tuple(sorted(TD_SIGNALS.keys())))

def _build_td_reference():
    prev = N.NeuralMass.DRIVE_EXPONENT
    try:
        N.NeuralMass.DRIVE_EXPONENT = _BETA
        if STATE["td_source"] == "simulator":
            mtd = N.NeuralMass(**N.TD_PARAMS)
            spec, pm, f, t = N.model_normspec(mtd, seed=43, n_runs=N.N_RUNS)
            return np.array([pm[b] for b in BANDS]), None
        profs = [_feats(s) for s in TD_SIGNALS.values()]
        return np.mean([p[0] for p in profs], axis=0), float(np.mean([p[1] for p in profs]))
    finally:
        N.NeuralMass.DRIVE_EXPONENT = prev

# ---- build / reuse TD reference ----
if STATE["td_source"] == "uploaded_pool" and len(TD_SIGNALS) == 0:
    print("✗ No TD recordings loaded and simulator not selected. Go back to Step 2.")
elif len(ASD_SIGNALS) == 0:
    print("✗ No ASD subjects loaded. Go back to Step 2.")
else:
    sig_now = _td_signature()
    if CACHE.get("TD_BANDS") is not None and CACHE.get("td_signature") == sig_now:
        print("✓ TD reference already cached — reusing it (no recompute).")
    else:
        print(f"Building TD reference from {('simulator' if STATE['td_source']=='simulator' else str(len(TD_SIGNALS))+' recordings')} … (once)")
        TD_BANDS, TD_EXP = _build_td_reference()
        CACHE["TD_BANDS"], CACHE["TD_EXP"], CACHE["td_signature"] = TD_BANDS, TD_EXP, sig_now

    # ---- pre-compute subject features (cached per subject) ----
    new = [nm for nm in ASD_SIGNALS if nm not in CACHE["subj_feats"]]
    if new:
        print(f"Pre-computing features for {len(new)} subject(s) …")
        for nm in new:
            CACHE["subj_feats"][nm] = _feats(ASD_SIGNALS[nm])
    # drop cached features for subjects no longer loaded
    for nm in list(CACHE["subj_feats"]):
        if nm not in ASD_SIGNALS:
            del CACHE["subj_feats"][nm]

    src = "TD simulator (least reliable)" if STATE["td_source"] == "simulator" else f"{len(TD_SIGNALS)} real TD recordings"
    print(f"\n✓ Engine: {STATE['engine']}   |   TD reference: {src}")
    print(f"  TD band profile (dB): {np.round(CACHE['TD_BANDS'],2)}")
    if CACHE["TD_EXP"] is not None:
        print(f"  TD aperiodic exponent: {CACHE['TD_EXP']:.3f}")
    print(f"  Subjects ready: {list(ASD_SIGNALS.keys())}")
    # coherence note: real EEG vs engine choice
    if STATE["td_source"] == "uploaded_pool":
        if STATE["engine"] == "classic":
            print("\nℹ  Note: you're comparing REAL EEG against a CLASSIC (aperiodically flat) reference.")
            print("   For a physically coherent comparison on the 1/f axis, consider the Realistic")
            print("   engine (re-run this cell after switching). Classic is best for reproducing the")
            print("   paper's validated numbers.")
        else:
            print("\nℹ  Note: Realistic engine — the reference and modulation responses carry the 1/f")
            print("   physics of real EEG, matching your recordings' scale. Good for interpreting")
            print("   modulations against real data.")
    print("\nNow use Step 4 (manual modulation) as many times as you like — it's instant.")
    print("Step 5 (recommendation) is optional and slower the first time.")

In [ ]:
#@title ⚡ Step 3b (optional): Surrogate accelerator — WHEN & WHY { display-mode: "form" }
#@markdown **What it is:** a CNN that *emulates* the simulator, ~2× faster per evaluation.
#@markdown
#@markdown **When to use it:** only if you plan **many** evaluations (large batches, parameter
#@markdown sweeps, or the AI component of the paper). For interactive use with a few subjects,
#@markdown **you do NOT need this** — the direct simulator in Steps 4–5 is fast enough and more
#@markdown accurate. This cell is placed *before* the evaluations on purpose, so that if you do
#@markdown train it, Steps 4–5 can use it.
#@markdown
#@markdown Training uses **resumable blocks** (a Colab restart never loses more than one block).
#@markdown Fidelity scales with samples (paper: R²≈0.93 at 5000 on the realistic engine).

run_training = False #@param {type:"boolean"}
n_samples = 1500 #@param {type:"slider", min:300, max:5000, step:100}
epochs = 40 #@param {type:"slider", min:10, max:80, step:5}

import os, glob, time, json, numpy as np

if not run_training:
    print("Accelerator NOT trained (default). Steps 4–5 use the direct simulator — recommended for")
    print("interactive use. Tick 'run_training' only if you need to accelerate large batches.")
else:
    # TensorFlow is only needed here (the optional accelerator). Colab ships a working build.
    # If it fails to import, the runtime's TF is corrupted (usually from a previous install of a
    # different TF over Colab's). We do NOT reinstall (that makes it worse); we stop cleanly and
    # tell you how to recover, without dumping a long traceback.
    _tf_ok = False
    try:
        import tensorflow as tf
        _tf_ok = True
        print(f"Using preinstalled TensorFlow {tf.__version__}")
    except Exception as e:
        print("⚠  TensorFlow could not be loaded in this runtime.")
        print("   This runtime's TensorFlow is corrupted (a previous install broke it).")
        print("   To fix it: menu → Runtime → Restart session (or Disconnect and delete runtime),")
        print("   then run the cells again from Step 1. Do NOT pip-install tensorflow.")
        print("   (You do NOT need the accelerator for interactive use — Steps 4–5 work without it.)")
        print(f"\n   Technical detail: {type(e).__name__}: {e}")

if run_training and _tf_ok:
    OUT = "/content/neuron_asd_train"; BLK = OUT+"/blocks"; os.makedirs(BLK, exist_ok=True)
    _BETA = 1.7 if STATE.get("engine")=="realistic" else None
    prev = N.NeuralMass.DRIVE_EXPONENT
    try:
        N.NeuralMass.DRIVE_EXPONENT = _BETA
        _, pow_td, _, _ = N.model_normspec(N.NeuralMass(**N.TD_PARAMS), seed=43, n_runs=N.N_RUNS)
    finally:
        N.NeuralMass.DRIVE_EXPONENT = prev

    done = sorted(glob.glob(BLK+"/block_*.npz"))
    have = sum(np.load(b)["X_mod"].shape[0] for b in done) if done else 0
    print(f"Resuming: {have}/{n_samples} samples in {len(done)} block(s).")
    t0=time.time(); bi=len(done); BLOCK=500
    while have < n_samples:
        k = min(BLOCK, n_samples-have); rng = np.random.default_rng(1000+bi)
        Xs,Xm,yd,yb = [],[],[],[]
        for i in range(k):
            baseline = N.ASD_PARAMS if rng.random()>0.5 else N.TD_PARAMS
            vec = np.concatenate([rng.uniform(-0.4,0.4,size=len(TARGETS)),
                                  rng.uniform(-1.0,1.0,size=len(NEUROMODS))])
            p = N.NeuralMass.DRIVE_EXPONENT
            try:
                N.NeuralMass.DRIVE_EXPONENT = _BETA
                model = N.build_model(baseline, vec)
                spec, pm, f, t = N.model_normspec(model, seed=bi*100000+i, n_runs=N.N_RUNS)
            finally:
                N.NeuralMass.DRIVE_EXPONENT = p
            Xs.append(np.asarray(spec,dtype=np.float32)); Xm.append(vec.astype(np.float32))
            yd.append(N.distance_to_td(pm, pow_td)); yb.append([pm.get(b,0.0) for b in BANDS])
        np.savez(f"{BLK}/block_{bi:03d}.npz", X_spec=np.array(Xs,dtype=np.float32),
                 X_mod=np.array(Xm,dtype=np.float32), y_dist=np.array(yd), y_bands=np.array(yb))
        have += k; bi += 1
        print(f"  [{time.time()-t0:.0f}s] block {bi} (+{k}) total={have}/{n_samples}")

    allb = sorted(glob.glob(BLK+"/block_*.npz"))
    Xs=np.concatenate([np.load(b)["X_spec"] for b in allb]); Xm=np.concatenate([np.load(b)["X_mod"] for b in allb])
    yd=np.concatenate([np.load(b)["y_dist"] for b in allb]); yb=np.concatenate([np.load(b)["y_bands"] for b in allb])
    rng=np.random.default_rng(7); idx=rng.permutation(len(Xs)); cut=int(0.8*len(Xs)); tr,te=idx[:cut],idx[cut:]
    print(f"Training CNN on {cut} samples ({len(Xs)-cut} held-out), {epochs} epochs …")
    ai=N.NeuronAI(); ai.train(Xs[tr],Xm[tr],yd[tr],yb[tr],epochs=int(epochs), model_out=OUT+"/surrogate.keras")
    fid=ai.evaluate_fidelity(Xs[te],Xm[te],yd[te])
    print(f"\n✓ Surrogate fidelity: R²={fid['r2']:.3f}  MAE={fid['mae_db']:.2f} dB  MAPE={fid['mape_pct']:.1f}%")
    if fid['r2'] < 0.85:
        print("  Tip: raise n_samples (e.g. 5000) for higher fidelity — it scales with data.")

In [ ]:
#@title 🔬 Step 4: Modulate manually → predicted effect + clinical-style note { display-mode: "form" }
#@markdown Set the subject and the modulations, then run. **Repeat as many times as you like** —
#@markdown change any control and re-run; it's instant (TD reference is cached).

subject = "ALL (cohort average)" #@param {type:"string"}
#@markdown ↑ Type an ASD subject name (from Step 2) or `ALL (cohort average)`.

#@markdown **Receptor / channel modulations** — Agonist / Inhibitor (translated exactly as in the paper)
GABA_A       = "None" #@param ["None", "Agonist", "Inhibitor"]
GABA_B       = "None" #@param ["None", "Agonist", "Inhibitor"]
NMDA_NR2A    = "None" #@param ["None", "Agonist", "Inhibitor"]
NMDA_NR2B    = "None" #@param ["None", "Agonist", "Inhibitor"]
AMPA         = "None" #@param ["None", "Agonist", "Inhibitor"]
D2_dopamine  = "None" #@param ["None", "Agonist", "Inhibitor"]
_5HT2A       = "None" #@param ["None", "Agonist", "Inhibitor"]
alpha7_nAChR = "None" #@param ["None", "Agonist", "Inhibitor"]

#@markdown **Global neuromodulators**
dopamine       = 0.0 #@param {type:"slider", min:-1, max:1, step:0.05}
serotonin      = 0.0 #@param {type:"slider", min:-1, max:1, step:0.05}
norepinephrine = 0.0 #@param {type:"slider", min:-1, max:1, step:0.05}
seed = 42 #@param {type:"slider", min:1, max:200, step:1}

import numpy as np, matplotlib.pyplot as plt

def _mod_vector():
    # Build the modulation EXACTLY as the engine does, so manual settings here match the
    # Step-5 recommendations: Agonist/Inhibitor are translated through each receptor's
    # agonist_sign x base_frac (e.g. a 5HT2A "Agonist" is a negative model input).
    sel = {"GABA_A":GABA_A, "GABA_B":GABA_B, "NMDA (NR2A)":NMDA_NR2A, "NMDA (NR2B)":NMDA_NR2B,
           "AMPA":AMPA, "D2_dopamine":D2_dopamine, "5HT2A":_5HT2A, "alpha7_nAChR":alpha7_nAChR}
    nmsel = {"Dopamine":dopamine, "Serotonin":serotonin, "Norepinephrine":norepinephrine}
    mod, neuromod = N.selections_to_mod(sel, nmsel)
    vec = N.mod_to_vector(mod, neuromod)
    active = {k: v for k, v in {**mod, **neuromod}.items() if abs(v) > 1e-9}
    return vec, active

def _predict(vec, subj_bands):
    """Predict the modulated profile the SAME way Step 5 evaluates moves, so manual results
    are coherent with the recommendation: apply the modulation's MEAN spectral response —
    measured over the stable reference baselines x seeds the recommender uses — to this
    subject's real profile. (A single-baseline simulation can flip sign for terrain-sensitive
    receptors; the ensemble-mean response is the quantity the paper's method optimizes.)"""
    prev = N.NeuralMass.DRIVE_EXPONENT
    try:
        N.NeuralMass.DRIVE_EXPONENT = (1.7 if STATE["engine"] == "realistic" else None)
        responses = []
        for bl in N.STABLE_REF_BASELINES:
            for sd in N.RESP_SEEDS:
                base = N._bands_vec(bl, np.zeros(len(N.MOD_ORDER)), sd, N.RESP_N_RUNS)
                responses.append(N._bands_vec(bl, vec, sd, N.RESP_N_RUNS) - base)
        mean_resp = np.mean(responses, axis=0)
        bands = np.asarray(subj_bands, float) + mean_resp
        # exponent shift: use the ASD-baseline simulation only for the E/I read-out direction
        model = N.build_model(dict(N.ASD_PARAMS), vec)
        FOOOF = N._lazy_fooof(); exp, _, _ = N._exp_of_signal(model.simulate_single(seed=int(seed)), FOOOF)
    finally:
        N.NeuralMass.DRIVE_EXPONENT = prev
    return bands, exp

def _dist(a,b): return float(np.sqrt(np.sum((np.asarray(a)-np.asarray(b))**2)))

# ---- guards ----
if CACHE.get("TD_BANDS") is None:
    print("✗ Build the TD reference first (run Step 3).")
elif len(ASD_SIGNALS) == 0:
    print("✗ No ASD subjects loaded (Step 2).")
else:
    td_bands = CACHE["TD_BANDS"]
    key = subject.strip()
    if key == "ALL (cohort average)" or key not in CACHE["subj_feats"]:
        feats = list(CACHE["subj_feats"].values())
        subj_bands = np.mean([f[0] for f in feats], axis=0)
        subj_exp   = float(np.mean([f[1] for f in feats]))
        subj_label = f"ASD cohort mean (n={len(feats)})"
        if key not in CACHE["subj_feats"] and key != "ALL (cohort average)":
            print(f"(subject '{key}' not found — showing cohort average; available: {list(ASD_SIGNALS)})")
    else:
        subj_bands, subj_exp = CACHE["subj_feats"][key]; subj_label = key

    vec, active = _mod_vector()
    if N._RESP_ENSEMBLE is None and not active:
        pass  # no modulation, no need to warn
    mod_bands, mod_exp = _predict(vec, subj_bands)
    d0, d1 = _dist(subj_bands, td_bands), _dist(mod_bands, td_bands)
    gain, gpct = d0-d1, (100*(d0-d1)/d0 if d0>1e-9 else 0.0)
    td_exp_ref = CACHE["TD_EXP"] if CACHE["TD_EXP"] is not None else (1.33 if STATE["engine"]=="realistic" else -0.28)

    # ---- figure ----
    fig = plt.figure(figsize=(11,6.2)); gs = fig.add_gridspec(2,2, height_ratios=[1.3,1.0])
    axA = fig.add_subplot(gs[0,:]); x=np.arange(len(BANDS)); w=0.26
    axA.bar(x-w, subj_bands, w, label=subj_label, color="#8894A6")
    axA.bar(x,   mod_bands,  w, label="After modulation", color="#2E7D6B")
    axA.bar(x+w, td_bands,   w, label="TD reference", color="#D9A404", alpha=0.85)
    axA.set_xticks(x); axA.set_xticklabels(BANDS, fontsize=12); axA.set_ylabel("relative power (dB)")
    axA.set_title("Spectral profile: subject → modulation → TD", fontweight="bold", loc="left")
    axA.legend(fontsize=9, ncol=3, loc="upper right"); axA.grid(axis="y", ls=":", alpha=0.5)
    for sp in ["top","right"]: axA.spines[sp].set_visible(False)
    axB = fig.add_subplot(gs[1,0]); bars=axB.bar(["Before","After"],[d0,d1],color=["#8894A6","#2E7D6B"],width=0.6)
    axB.set_ylabel("distance to TD (dB)"); axB.set_title("Distance to TD", fontweight="bold", loc="left")
    for b,v in zip(bars,[d0,d1]): axB.text(b.get_x()+b.get_width()/2,v,f"{v:.2f}",ha="center",va="bottom",fontsize=11)
    axB.grid(axis="y", ls=":", alpha=0.5)
    for sp in ["top","right"]: axB.spines[sp].set_visible(False)
    axC = fig.add_subplot(gs[1,1])
    axC.axvline(subj_exp, color="#8894A6", lw=2.5, label=f"subject ({subj_exp:.2f})")
    axC.axvline(mod_exp,  color="#2E7D6B", lw=2.5, label=f"modulated ({mod_exp:.2f})")
    axC.axvline(td_exp_ref, color="#D9A404", lw=2.5, ls="--", label=f"TD ref ({td_exp_ref:.2f})")
    axC.set_xlim(min(subj_exp,mod_exp,td_exp_ref)-0.4, max(subj_exp,mod_exp,td_exp_ref)+0.4)
    axC.set_yticks([]); axC.set_xlabel("aperiodic exponent (steeper → lower E/I)")
    axC.set_title("E/I placement", fontweight="bold", loc="left")
    axC.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5,-0.35))
    for sp in ["top","right","left"]: axC.spines[sp].set_visible(False)
    col = "#2E7D6B" if gain>0 else "#C0392B"
    verdict = "moves TOWARD TD ✓" if gain>0 else "moves AWAY from TD ✗"
    fig.suptitle(f"Predicted effect: {verdict}   (Δ distance = {gain:+.2f} dB, {gpct:+.1f}%)",
                 fontsize=13, fontweight="bold", color=col, y=1.00)
    fig.subplots_adjust(hspace=0.45, wspace=0.28, top=0.90, bottom=0.16, left=0.08, right=0.97)
    plt.show()

    # ---- clinical-style note (printed, not a file) ----
    band_gap = subj_bands - td_bands
    lead = BANDS[int(np.argmax(np.abs(band_gap)))]
    ei_word = ("above (lower exponent → higher E/I)" if subj_exp > td_exp_ref
               else "below (higher exponent → lower E/I)")
    mod_txt = ", ".join(f"{k} {'+' if v>0 else '−'}{abs(v):.2f}" for k,v in active.items()) or "no modulation set"
    print("\n" + "─"*70)
    print(f"CLINICAL-STYLE NOTE · {subj_label}   [engine: {STATE['engine']}"
          + ("; TD=simulator, least reliable" if STATE['td_source']=='simulator' else "]"))
    print("─"*70)
    print(f"• Baseline: this subject sits {ei_word} the TD reference on the E/I axis")
    print(f"  (subject exponent {subj_exp:.2f} vs TD {td_exp_ref:.2f}); the largest spectral")
    print(f"  deviation is in the {lead} band ({band_gap[BANDS.index(lead)]:+.1f} dB vs TD).")
    print(f"• Applied modulation: {mod_txt}")
    if gain > 0:
        print(f"• Predicted effect: moves the profile TOWARD TD, closing {gpct:.0f}% of the gap")
        print(f"  (distance {d0:.2f} → {d1:.2f} dB). This modulation is a promising hypothesis")
        print(f"  for this subject; try nearby settings to see if the gain increases.")
    elif abs(gain) < 0.05*max(d0,1e-9):
        print(f"• Predicted effect: essentially no change (distance {d0:.2f} → {d1:.2f} dB).")
        print(f"  Consider a different target or direction.")
    else:
        print(f"• Predicted effect: moves the profile AWAY from TD (distance {d0:.2f} → {d1:.2f} dB).")
        print(f"  The opposite direction on the dominant target is worth trying.")
    print("• For the model's own best suggestion, run Step 5 (per-subject recommendation).")
    if STATE["td_source"] == "uploaded_pool" and STATE["engine"] == "classic":
        print("• Engine note: results shown against a CLASSIC (aperiodically flat) reference. For a")
        print("  1/f-coherent comparison with real EEG, switch to the Realistic engine in Step 3.")
    print("• Reminder: model-based prediction for hypothesis generation, not a clinical prescription.")
    print("─"*70)

In [ ]:
#@title 🎯 Step 5 (optional): Per-subject recommendation + clinical note { display-mode: "form" }
#@markdown The model's own best modulations for each subject, via the validated response-projection.
#@markdown The response ensemble is built **once** (~3–4 min the first time) and then **cached**, so
#@markdown subsequent runs — including changing the subject — are fast.

which_subject = "ALL subjects" #@param {type:"string"}
#@markdown ↑ `ALL subjects` or a specific name from Step 2.
top_k = 4 #@param {type:"slider", min:1, max:8, step:1}

import numpy as np

if CACHE.get("TD_BANDS") is None:
    print("✗ Build the TD reference first (Step 3).")
elif len(ASD_SIGNALS) == 0:
    print("✗ No ASD subjects loaded (Step 2).")
else:
    td_bands = CACHE["TD_BANDS"]
    if N._RESP_ENSEMBLE is None:
        print("Building the response ensemble once (~3–4 min; cached afterwards) …")
    else:
        print("Using cached response ensemble (fast).")

    def _report(nm):
        subj_bands, subj_exp = CACHE["subj_feats"][nm]
        recs, n = N.recommend_by_projection(subj_bands, td_bands, top_k=int(top_k), gate=N.RESP_NOISE_GATE)
        # only report moves selected above the stability threshold — these are the ones that
        # individually move the subject toward TD and are reproducible in Step 4. Lower-confidence
        # moves are selected only in combination and can look contrary on their own.
        thr = getattr(N, "REC_STABILITY_THRESHOLD", 0.6)
        strong = [(name, dirn, conf) for (name, dirn, conf) in recs if conf >= thr]
        td_exp_ref = CACHE["TD_EXP"] if CACHE["TD_EXP"] is not None else (1.33 if STATE["engine"]=="realistic" else -0.28)
        print("\n" + "═"*70)
        print(f"RECOMMENDATION · {nm}   [engine: {STATE['engine']}"
              + ("; TD=simulator, least reliable]" if STATE['td_source']=='simulator' else "]"))
        print("═"*70)
        if not strong:
            print(f"• No modulation exceeds the reliability threshold ({int(thr*100)}%) for this subject —")
            print("  a valid outcome: the subject is already close to TD, or no single move reliably helps.")
            if recs:
                print(f"  (Highest-confidence candidate below threshold: {recs[0][0]} {recs[0][1]}, "
                      f"{recs[0][2]*100:.0f}%.)")
        else:
            print(f"• Reliable modulations (confidence ≥ {int(thr*100)}%, reproducible in Step 4):")
            for i,(name,dirn,conf) in enumerate(strong):
                print(f"    {i+1}. {name:14s} {dirn:10s}  (confidence {conf*100:.0f}%)")
            top = strong[0]
            print(f"• Clinical-style summary: for {nm}, the model's strongest reliable hypothesis is")
            print(f"  **{top[0]} — {top[1]}** (confidence {top[2]*100:.0f}%). Applying it in Step 4")
            print(f"  moves this subject's profile toward TD. Lower-confidence moves are omitted")
            print(f"  because they only help in combination and may look contrary on their own.")
        print("• Reminder: model-based prediction for hypothesis generation, not a prescription.")
        print("═"*70)

    names = list(ASD_SIGNALS.keys()) if which_subject.strip()=="ALL subjects" else \
            ([which_subject.strip()] if which_subject.strip() in ASD_SIGNALS else [])
    if not names:
        print(f"✗ Subject '{which_subject}' not found. Available: {list(ASD_SIGNALS.keys())}")
    else:
        for nm in names:
            _report(nm)